# Customer Churn Prediction using Machine Learning

## Project Objective
The objective of this project is to predict whether a telecom customer is likely to **churn (leave the company)** based on customer information, services, contract details, payment method, tenure, and billing information.

## Machine Learning Algorithm
**Logistic Regression** is used because this is a binary classification problem:
- `0` = Customer will not churn
- `1` = Customer will churn

## Project Steps
1. Import libraries
2. Load the dataset
3. Understand the dataset
4. Perform data cleaning
5. Exploratory Data Analysis (EDA)
6. Prepare features and target
7. Encode categorical variables
8. Split data into training and testing sets
9. Scale numerical features
10. Train Logistic Regression model
11. Evaluate the model
12. Predict churn for a new customer

In [ ]:
# Step 1: Import Libraries

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay
)

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

## Step 2: Load the Dataset

The dataset used in this project is the **Telco Customer Churn dataset**.

In [ ]:
# Load dataset
file_path = r"/mnt/data/WA_Fn-UseC_-Telco-Customer-Churn(1).csv"
df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()

In [ ]:
# Step 3: Basic Information

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

In [ ]:
# Step 4: Statistical Summary

df.describe(include="all").T

In [ ]:
# Step 5: Check Missing Values

missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

## Step 6: Data Cleaning

`TotalCharges` can contain blank values in this dataset. We convert it to numeric so that blank values become missing values and can be handled safely by the ML pipeline.

`customerID` is only an identifier, so it is removed before modelling.

In [ ]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove customer ID because it does not help predict churn
df = df.drop(columns=["customerID"])

print("Shape after removing customerID:", df.shape)
print("Missing TotalCharges:", df["TotalCharges"].isna().sum())

# Exploratory Data Analysis (EDA)

EDA helps us understand customer behaviour before building the machine learning model.

In [ ]:
# Churn distribution

print(df["Churn"].value_counts())
print("\nChurn percentage:")
print((df["Churn"].value_counts(normalize=True) * 100).round(2))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
# Churn by Contract

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Customer Churn by Contract Type")
plt.xticks(rotation=15)
plt.show()

In [ ]:
# Churn by Internet Service

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="InternetService", hue="Churn")
plt.title("Customer Churn by Internet Service")
plt.show()

In [ ]:
# Tenure distribution by churn

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Tenure vs Customer Churn")
plt.show()

In [ ]:
# Monthly Charges distribution by churn

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges vs Customer Churn")
plt.show()

## Key EDA Observations

- Customers with **month-to-month contracts** generally show more churn.
- Customers with **longer tenure** generally show lower churn.
- Higher monthly charges can be associated with greater churn.
- Contract type, internet service, payment method, and other services can provide useful information for prediction.

# Step 7: Prepare Features and Target

Target variable:
- `Churn`

All other useful customer attributes are used as input features.

In [ ]:
# Separate input features and target

X = df.drop(columns=["Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget values:")
print(y.value_counts())

In [ ]:
# Identify numerical and categorical columns

numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

# Step 8: Train-Test Split

We use:
- **80% data for training**
- **20% data for testing**

`stratify=y` keeps the churn class proportion similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

# Step 9: Preprocessing

Machine learning algorithms need numerical input.

We use:
- **Median imputation** for numerical missing values
- **StandardScaler** for numerical features
- **Most frequent imputation** for categorical missing values
- **OneHotEncoder** for categorical variables

In [ ]:
# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

# Step 10: Build the Machine Learning Model

We use **Logistic Regression**.

Logistic Regression is commonly used for binary classification problems such as:
- Yes / No
- Churn / Not Churn
- Fraud / Not Fraud
- Pass / Fail

In [ ]:
# Create complete ML pipeline

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# Train model
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

# Step 11: Model Prediction

In [ ]:
# Make predictions

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("First 20 actual values:")
print(y_test.iloc[:20].values)

print("\nFirst 20 predicted values:")
print(y_pred[:20])

# Step 12: Model Evaluation

In [ ]:
# Accuracy

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

In [ ]:
# Classification Report

print(classification_report(
    y_test,
    y_pred,
    target_names=["No Churn", "Churn"]
))

In [ ]:
# Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# ROC-AUC Score

roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.3f}")

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve - Customer Churn")
plt.show()

# Step 13: Test the Model with a New Customer

Below is an example customer. The same columns used during training are provided to the model.

The model returns:
- Prediction: Churn / No Churn
- Probability of churn

In [ ]:
# Example new customer

new_customer = pd.DataFrame([{
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 12,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 75.50,
    "TotalCharges": 906.00
}])

prediction = model.predict(new_customer)[0]
probability = model.predict_proba(new_customer)[0, 1]

print("Prediction:", "Churn" if prediction == 1 else "No Churn")
print(f"Churn Probability: {probability:.2%}")

# Step 14: Save the Trained Model

The trained pipeline contains preprocessing and Logistic Regression together.
It can be saved and later loaded into a Streamlit application.

In [ ]:
import joblib

model_file = "churn_model.pkl"
joblib.dump(model, model_file)

print(f"Model saved successfully as: {model_file}")

# Final Conclusion

This project uses **Machine Learning to predict customer churn**.

### What we learned
- How to load and understand a real-world dataset
- How to clean missing values
- How to perform basic EDA
- How to handle categorical variables using One-Hot Encoding
- Why numerical features are scaled
- How to split data into training and testing sets
- How to train Logistic Regression
- How to evaluate a classification model
- How to make a prediction for a new customer
- How to save a trained model for use in Streamlit

### Project Flow

**Dataset → Data Cleaning → EDA → Feature Preparation → Encoding & Scaling → Train/Test Split → Logistic Regression → Evaluation → Prediction → Streamlit**

### Simple explanation for teacher

> "My project is Customer Churn Prediction. I used the Telco Customer Churn dataset. First, I cleaned the data and performed EDA to understand customer behaviour. Then I converted categorical data using one-hot encoding and scaled numerical features. I used Logistic Regression because churn is a binary classification problem. Finally, I evaluated the model using accuracy, classification report, confusion matrix and ROC-AUC, and used the trained model to predict whether a new customer is likely to churn."